# Pemetaan Potensi Kawasan PLTPs (Pembangkit Listrik Tenaga Pasang Surut)
### Studi Kasus: Perairan Sulawesi Utara

Notebook ini mengklasifikasikan **potensi energi pasang surut** suatu kawasan perairan ke dalam 3 kelas
(*Rendah, Sedang, Tinggi*), dengan menggabungkan empat sumber data geospasial:

| Sumber Data | Format Asli | Info yang Diambil |
|---|---|---|
| Citra satelit batimetri (BATNAS) | GeoTIFF | Kedalaman laut |
| Data arus laut (CMEMS) | NetCDF | Kecepatan arus pasang surut |
| Observasi & prediksi pasang surut (BMKG: Bitung, Likupang, Manado) | CSV | Rentang pasang surut (tidal range) |
| Jarak ke gardu induk terdekat | CSV | Jarak infrastruktur listrik |

**Alur kerja:**
1. Konversi data mentah (raster & NetCDF) menjadi tabel.
2. Rekayasa fitur per sumber data (tidal range, kecepatan arus, koordinat piksel batimetri).
3. Menyatukan seluruh sumber data secara spasial (*spatial join*) berdasarkan titik grid terdekat.
4. Membentuk target `potensi_kawasan` dari skor energi gabungan.
5. Melatih & membandingkan 3 model klasifikasi (Random Forest, XGBoost, SVM).
6. Menjelaskan model dengan Permutation Feature Importance (PFI) dan SHAP.

> **Catatan:** notebook ini berjalan di Google Colab dan membaca data mentah dari Google Drive
> (folder `dataset-data-mining`).


## 1. Setup & Import Library

In [ ]:
!pip install -q xarray netCDF4 rasterio contextily shap

In [ ]:
# =========================
# Data Manipulation
# =========================
import numpy as np
import pandas as pd
import xarray as xr

# =========================
# Geospatial & Raster Data
# =========================
import rasterio
import geopandas as gpd
import contextily as cx
from PIL import Image

# =========================
# Machine Learning
# =========================
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, cohen_kappa_score, confusion_matrix,
)

# =========================
# Explainable AI
# =========================
import shap

# =========================
# Visualization
# =========================
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

### Mount Google Drive\nSeluruh file data mentah disimpan di Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Konfigurasi

Semua path file dan konstanta dikumpulkan di satu tempat agar mudah disesuaikan tanpa perlu
menelusuri seluruh notebook.


In [ ]:
BASE_DIR = "/content/drive/MyDrive/dataset-data-mining"

PATHS = {
    "batnas_tif": f"{BASE_DIR}/BATNAS_120E-125E_000-05N_MSL_v1.1.tif",
    "arus_nc": f"{BASE_DIR}/cmems_mod_glo_phy_anfc_merged-uv_PT1H-i_1753806523157.nc",
    "obs_bitung": f"{BASE_DIR}/Bitung.csv",
    "obs_likupang": f"{BASE_DIR}/Likupang.csv",
    "obs_manado": f"{BASE_DIR}/Manado.csv",
    "pred_bitung": f"{BASE_DIR}/pasang_surut_bitung.csv",
    "pred_likupang": f"{BASE_DIR}/pasang_surut_Likupang.csv",
    "pred_manado": f"{BASE_DIR}/pasang_surut_manado.csv",
    "jarak_grid": f"{BASE_DIR}/proximity_to_grid_sulut_preprocesing.csv",
}

# Koordinat referensi tiga stasiun observasi BMKG (lat, lon)
COORDINATE_MAP = {
    "Bitung": (1.4589, 125.2178),
    "Likupang": (1.6986, 125.0144),
    "Manado": (1.4978, 124.8394),
}

UTM_CRS = "EPSG:32651"   # Zona UTM 51N, mencakup wilayah Sulawesi Utara
RANDOM_STATE = 42

## 3. Konversi Data Mentah (Raster & NetCDF) ke Tabular

Data batimetri (kedalaman laut) datang dalam format GeoTIFF, sedangkan data arus laut dalam format
NetCDF. Keduanya perlu dikonversi ke `DataFrame` agar bisa diproses bersama data tabular lainnya.


In [ ]:
def convert_tif_to_dataframe(tif_path: str) -> pd.DataFrame:
    """Konversi raster GeoTIFF menjadi DataFrame piksel (baris x kolom -> nilai)."""
    img = Image.open(tif_path)
    img_array = np.array(img)

    if img_array.ndim == 2:  # citra grayscale (mis. peta kedalaman 1 band)
        return pd.DataFrame(img_array)

    h, w, c = img_array.shape  # citra multi-band
    reshaped = img_array.reshape((h * w, c))
    return pd.DataFrame(reshaped, columns=[f"Channel_{i}" for i in range(c)])


def convert_netcdf_tide_to_dataframe(nc_path: str) -> pd.DataFrame:
    """Konversi NetCDF arus laut (variabel utide, vtide) menjadi DataFrame permukaan."""
    ds = xr.open_dataset(nc_path)
    ds_uv = xr.merge([ds["utide"], ds["vtide"]])
    df = ds_uv.to_dataframe().reset_index()

    if "depth" in df.columns:  # ambil hanya lapisan permukaan
        df = df[df["depth"] == df["depth"].min()]

    return df

In [ ]:
# Konversi batimetri Sulawesi Utara (GeoTIFF -> CSV)
df_batnas = convert_tif_to_dataframe(PATHS["batnas_tif"])
print(f"Batimetri (grid piksel): {df_batnas.shape}")

# Konversi arus laut Sulawesi Utara (NetCDF -> CSV)
df_tide_current = convert_netcdf_tide_to_dataframe(PATHS["arus_nc"])
print(f"Arus laut: {df_tide_current.shape}")

## 4. Memuat Dataset Observasi, Prediksi, dan Jarak Grid

| DataFrame | Deskripsi |
|---|---|
| `df_bitung`, `df_likupang`, `df_manado` | Observasi pasang surut aktual per lokasi (`Date`, `EST`, `MSL`, `LAT`) |
| `df_pasang_bitung`, `df_pasang_likupang`, `df_pasang_manado` | Prediksi pasang/surut per waktu (`Waktu`, `Ketinggian (m)`, `Jenis`) |
| `df_batnas` | Grid kedalaman laut (dari konversi GeoTIFF) |
| `df_tide_current` | Kecepatan arus laut permukaan (dari konversi NetCDF): `utide`, `vtide` |
| `df_jarak` | Jarak tiap titik ke gardu induk terdekat |


In [ ]:
df_bitung = pd.read_csv(PATHS["obs_bitung"])
df_likupang = pd.read_csv(PATHS["obs_likupang"])
df_manado = pd.read_csv(PATHS["obs_manado"])

df_pasang_bitung = pd.read_csv(PATHS["pred_bitung"])
df_pasang_likupang = pd.read_csv(PATHS["pred_likupang"])
df_pasang_manado = pd.read_csv(PATHS["pred_manado"])

df_jarak = pd.read_csv(PATHS["jarak_grid"])

print(f"Observasi  -> Bitung: {df_bitung.shape}, Likupang: {df_likupang.shape}, Manado: {df_manado.shape}")
print(f"Prediksi   -> Bitung: {df_pasang_bitung.shape}, Likupang: {df_pasang_likupang.shape}, Manado: {df_pasang_manado.shape}")
print(f"Batimetri  -> {df_batnas.shape}")
print(f"Arus laut  -> {df_tide_current.shape}")
print(f"Jarak grid -> {df_jarak.shape}")

## 5. Pengecekan Kualitas Data

In [ ]:
raw_dataframes = {
    "Observasi Bitung": df_bitung,
    "Observasi Likupang": df_likupang,
    "Observasi Manado": df_manado,
    "Prediksi Bitung": df_pasang_bitung,
    "Prediksi Likupang": df_pasang_likupang,
    "Prediksi Manado": df_pasang_manado,
    "Batimetri (raw grid)": df_batnas,
    "Arus Laut": df_tide_current,
    "Jarak ke Grid": df_jarak,
}

for name, df in raw_dataframes.items():
    n_missing = df.isnull().sum().sum()
    print(f"{name:<22}: shape={df.shape}, total nilai kosong={n_missing}")

## 6. Rekayasa Fitur (Feature Engineering)

### 6.1 Rentang Pasang Surut (Tidal Range) per 6 Jam

Untuk setiap lokasi, data prediksi pasang-surut dikelompokkan ke interval 6 jam, lalu dihitung selisih
antara titik pasang tertinggi dan surut terendah pada interval tersebut (*tidal range*). Nilai `LAT`
(Lowest Astronomical Tide) dari data observasi digabungkan sebagai referensi elevasi.


In [ ]:
def hitung_rentang_prediksi_per_6jam(
    df, lat_df, lokasi_nama,
    waktu_col="Waktu", tinggi_col="Ketinggian (m)", jenis_col="Jenis",
):
    """
    Menghitung rentang pasang-surut per interval 6 jam dan menggabungkan LAT dari data observasi.

    Args:
        df (pd.DataFrame): Data prediksi pasang-surut (df_pasang_xxx).
        lat_df (pd.DataFrame): Data observasi dengan kolom Date dan LAT.
        lokasi_nama (str): Nama lokasi.

    Returns:
        pd.DataFrame: waktu interval 6 jam, tidal range, LAT, dan lokasi.
    """
    temp_df = df.copy()
    temp_df[waktu_col] = pd.to_datetime(temp_df[waktu_col])
    temp_df["waktu_6jam"] = temp_df[waktu_col].dt.floor("6H")

    df_pasang = temp_df[temp_df[jenis_col] == "Pasang"]
    df_surut = temp_df[temp_df[jenis_col] == "Surut"]

    max_6jam_pasang = df_pasang.groupby("waktu_6jam")[tinggi_col].max()
    min_6jam_surut = df_surut.groupby("waktu_6jam")[tinggi_col].min()

    interval_df = pd.DataFrame({
        "ketinggian_max_pasang": max_6jam_pasang,
        "ketinggian_min_surut": min_6jam_surut,
    }).dropna()
    interval_df["tidal_range_6jam"] = (
        interval_df["ketinggian_max_pasang"] - interval_df["ketinggian_min_surut"]
    )

    lat_df = lat_df.copy()
    lat_df["Date"] = pd.to_datetime(lat_df["Date"])
    lat_df["waktu_6jam"] = lat_df["Date"].dt.floor("6H")
    lat_per_6jam = lat_df.groupby("waktu_6jam")["LAT"].mean()

    hasil = interval_df.join(lat_per_6jam, on="waktu_6jam")
    hasil["lokasi"] = lokasi_nama

    return hasil.reset_index()[["waktu_6jam", "tidal_range_6jam", "LAT", "lokasi"]]

In [ ]:
df_tidal_range_lat = pd.concat([
    hitung_rentang_prediksi_per_6jam(df_pasang_bitung, df_bitung, "Bitung"),
    hitung_rentang_prediksi_per_6jam(df_pasang_likupang, df_likupang, "Likupang"),
    hitung_rentang_prediksi_per_6jam(df_pasang_manado, df_manado, "Manado"),
], ignore_index=True)

df_tidal_range_lat.info()
df_tidal_range_lat.describe()

### 6.2 Kecepatan Arus Pasang Surut

Kecepatan arus total dihitung dari komponen `utide` (timur-barat) dan `vtide` (utara-selatan) menggunakan magnitudo vektor.

In [ ]:
df_kecepatan_arus = df_tide_current.copy()
df_kecepatan_arus[["utide", "vtide"]] = df_kecepatan_arus[["utide", "vtide"]].fillna(0)

# Kecepatan total = akar(utide^2 + vtide^2)
df_kecepatan_arus["total_tidal_speed"] = np.sqrt(
    df_kecepatan_arus["utide"] ** 2 + df_kecepatan_arus["vtide"] ** 2
)

df_kecepatan_arus.describe(include="all")

### 6.3 Batimetri: Unpivot Grid & Konversi ke Koordinat Geografis

Grid piksel hasil konversi GeoTIFF (baris x kolom) di-*unpivot* menjadi format panjang (satu baris per
piksel), disaring agar hanya menyisakan piksel laut (`depth < 0`), lalu indeks baris/kolomnya dikonversi
menjadi koordinat latitude/longitude menggunakan geotransform dari file `.tif` aslinya.


In [ ]:
df_long = df_batnas.reset_index().rename(columns={"index": "row"})
df_tidy = df_long.melt(id_vars=["row"], var_name="col", value_name="depth")
df_tidy["col"] = df_tidy["col"].astype(int)

# Hanya simpan piksel laut (kedalaman negatif)
df_sea = df_tidy[df_tidy["depth"] < 0].copy()
print(f"Jumlah piksel awal: {len(df_tidy)} -> setelah filter daratan: {len(df_sea)}")

In [ ]:
with rasterio.open(PATHS["batnas_tif"]) as src:
    geotransform = src.transform

longitudes, latitudes = rasterio.transform.xy(
    transform=geotransform,
    rows=df_sea["row"].values,
    cols=df_sea["col"].values,
    offset="center",
)
df_sea["longitude"] = longitudes
df_sea["latitude"] = latitudes

df_final_batnas = df_sea[["latitude", "longitude", "depth"]].reset_index(drop=True)
df_final_batnas.info()

### 6.4 Jarak ke Gardu Induk Terdekat

Koordinat gardu induk dikonversi dari format terskala (mikro-derajat) ke derajat desimal.

In [ ]:
df_jarak = df_jarak.rename(columns={"Nama Lokasi": "Lokasi"})
df_jarak["latitude_gi"] = df_jarak["Lat_G"] / 1_000_000
df_jarak["longitude_gi"] = df_jarak["Long_G"] / 1_000_000

df_gardu_induk = df_jarak[["Lokasi", "latitude_gi", "longitude_gi"]]
df_gardu_induk

## 7. Menyatukan Seluruh Sumber Data secara Spasial

Keempat sumber data (kecepatan arus, kedalaman, tidal range, jarak grid) berada pada grid/titik koordinat
yang berbeda-beda. Untuk menyatukannya, tiap grid kecepatan arus dicocokkan dengan **titik terdekat**
dari sumber data lain menggunakan *spatial nearest join* (`geopandas.sjoin_nearest`), setelah seluruh
data diproyeksikan ke sistem koordinat UTM (satuan meter) agar jarak terukur akurat.


### 7.1 Agregasi per Lokasi/Grid

In [ ]:
df_lokasi = pd.DataFrame(list(COORDINATE_MAP.items()), columns=["lokasi", "coordinates"])
df_lokasi[["latitude", "longitude"]] = pd.DataFrame(
    df_lokasi["coordinates"].tolist(), index=df_lokasi.index
)
df_lokasi = df_lokasi.drop(columns="coordinates")

# Kecepatan arus maksimum per titik grid
max_speed_per_location = (
    df_kecepatan_arus.groupby(["latitude", "longitude"])["total_tidal_speed"]
    .max()
    .reset_index()
    .rename(columns={"total_tidal_speed": "max_tidal_speed"})
)

# Rata-rata tidal range per stasiun BMKG
avg_range_per_location = (
    df_tidal_range_lat.groupby("lokasi")["tidal_range_6jam"]
    .mean()
    .reset_index()
    .rename(columns={"tidal_range_6jam": "mean_tidal_range"})
)

### 7.2 Membentuk GeoDataFrame & Reprojeksi ke UTM

In [ ]:
gdf_kecepatan = gpd.GeoDataFrame(
    max_speed_per_location,
    geometry=gpd.points_from_xy(max_speed_per_location.longitude, max_speed_per_location.latitude),
    crs="EPSG:4326",
)

gdf_batnas = gpd.GeoDataFrame(
    df_final_batnas,
    geometry=gpd.points_from_xy(df_final_batnas.longitude, df_final_batnas.latitude),
    crs="EPSG:4326",
)

df_tidal_range_berkoordinat = pd.merge(avg_range_per_location, df_lokasi, on="lokasi")
gdf_tidal_range = gpd.GeoDataFrame(
    df_tidal_range_berkoordinat,
    geometry=gpd.points_from_xy(df_tidal_range_berkoordinat.longitude, df_tidal_range_berkoordinat.latitude),
    crs="EPSG:4326",
)

gdf_gardu_induk = gpd.GeoDataFrame(
    df_gardu_induk,
    geometry=gpd.points_from_xy(df_gardu_induk.longitude_gi, df_gardu_induk.latitude_gi),
    crs="EPSG:4326",
)

# Reprojeksi ke UTM Zona 51N agar jarak nearest-join terukur dalam meter, bukan derajat
gdf_kecepatan_utm = gdf_kecepatan.to_crs(UTM_CRS)
gdf_batnas_utm = gdf_batnas.to_crs(UTM_CRS)
gdf_tidal_range_utm = gdf_tidal_range.to_crs(UTM_CRS)
gdf_gardu_induk_utm = gdf_gardu_induk.to_crs(UTM_CRS)

### 7.3 Spatial Join Bertahap

In [ ]:
# 1) Grid kecepatan arus + kedalaman (batimetri) terdekat
print("Menggabungkan data kecepatan dan batimetri...")
gdf_master_grid = gpd.sjoin_nearest(gdf_kecepatan_utm, gdf_batnas_utm, how="left")
gdf_master_grid = gdf_master_grid.drop(columns=["index_right"], errors="ignore")

# Kolom koordinat kecepatan (kiri) & batimetri (kanan) otomatis diberi suffix _left / _right
# oleh geopandas karena kedua sumber sama-sama punya kolom 'latitude'/'longitude'.
gdf_master_grid = gdf_master_grid.rename(columns={
    "latitude_left": "latitude", "longitude_left": "longitude",
})

# 2) + rentang pasang surut dari stasiun BMKG terdekat
print("Menambahkan informasi tidal range dari stasiun terdekat...")
final_dataset_gdf = gpd.sjoin_nearest(gdf_master_grid, gdf_tidal_range_utm, how="left")
final_dataset_gdf = final_dataset_gdf.drop(
    columns=["index_right", "latitude", "longitude"], errors="ignore"
)  # kolom lat/lon di sini milik stasiun BMKG, bukan grid utama -> tidak diperlukan

final_dataset_gdf.head()

In [ ]:
# 3) + jarak ke gardu induk terdekat
print("Menambahkan informasi jarak ke gardu induk terdekat...")
dataset_lengkap_gdf = gpd.sjoin_nearest(
    final_dataset_gdf, gdf_gardu_induk_utm, how="left", distance_col="distance_to_grid_m",
)
dataset_lengkap_gdf.info()

### 7.4 Pembersihan Akhir untuk Machine Learning

In [ ]:
dataset_lengkap_gdf["distance_to_grid_km"] = dataset_lengkap_gdf["distance_to_grid_m"] / 1000

# Kolom identitas gardu/join tidak diperlukan untuk pemodelan, tapi 'latitude_right'/'longitude_right'
# (koordinat titik batimetri terdekat) disimpan sebagai referensi stasiun untuk keperluan dokumentasi.
final_df_for_ml = pd.DataFrame(dataset_lengkap_gdf.drop(columns=["geometry"], errors="ignore"))
final_df_for_ml = final_df_for_ml.rename(columns={
    "latitude_right": "latitude_batimetri",
    "longitude_right": "longitude_batimetri",
})
final_df_for_ml = final_df_for_ml.drop(
    columns=["index_right", "Lokasi", "latitude_gi", "longitude_gi", "distance_to_grid_m"],
    errors="ignore",
)

print("=== Dataset Final Siap untuk Model ===")
final_df_for_ml.info()

## 8. Rekayasa Fitur Target: Skor Energi & Kelas Potensi Kawasan

Skor potensi energi dihitung dari kecepatan arus (dipangkatkan 3, sesuai rumus daya kinetik arus)
dikalikan rentang pasang surut, lalu disesuaikan dengan penalti jarak ke grid listrik. Skor ini
dinormalisasi dan dibagi menjadi 3 kelas seimbang (`Rendah`, `Sedang`, `Tinggi`) menggunakan kuantil.


In [ ]:
# Skor energi = kecepatan_arus^3 * rentang_pasang_surut  (proporsional terhadap daya kinetik arus)
final_df_for_ml["energy_score"] = (
    final_df_for_ml["max_tidal_speed"] ** 3 * final_df_for_ml["mean_tidal_range"]
)

# Penalti jarak: lokasi yang jauh dari grid listrik mendapat skor lebih rendah
final_df_for_ml["adjusted_score"] = (
    final_df_for_ml["energy_score"] / (1 + final_df_for_ml["distance_to_grid_km"])
)

# Normalisasi ke rentang [0, 1]
scaler = MinMaxScaler()
final_df_for_ml["adjusted_score_norm"] = scaler.fit_transform(final_df_for_ml[["adjusted_score"]])

# Bagi menjadi 3 kelas potensi dengan jumlah sampel seimbang (kuantil)
final_df_for_ml["potensi_kawasan"] = pd.qcut(
    final_df_for_ml["adjusted_score_norm"], q=3, labels=["Rendah", "Sedang", "Tinggi"]
)

print(final_df_for_ml["potensi_kawasan"].value_counts())
final_df_for_ml.isnull().sum()

### 8.1 Eksplorasi Korelasi Antar Fitur

In [ ]:
kolom_skor = ["energy_score", "adjusted_score", "adjusted_score_norm"]
correlation_matrix = (
    final_df_for_ml.select_dtypes(include=np.number)
    .drop(columns=kolom_skor, errors="ignore")
    .corr()
)

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriks Korelasi Fitur (di luar kolom skor)")
plt.show()

## 9. Pembagian Data & Encoding Label

Fitur prediktor yang dipakai untuk pemodelan adalah `max_tidal_speed`, `depth`, `mean_tidal_range`, dan
`distance_to_grid_km` — representasi ringkas dari keempat sumber data. Target `potensi_kawasan` di-encode
menjadi numerik (0=Rendah, 1=Sedang, 2=Tinggi), lalu data dibagi 80:20 dengan stratifikasi agar proporsi
kelas tetap seimbang di data latih maupun data uji.


In [ ]:
FEATURE_COLUMNS = ["max_tidal_speed", "depth", "mean_tidal_range", "distance_to_grid_km"]

X = final_df_for_ml[FEATURE_COLUMNS]
y = final_df_for_ml["potensi_kawasan"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train_encoded, y_test_encoded = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded,
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print("\nLabel Encoding:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label}: {i}")

## 10. Pelatihan Model & Hyperparameter Tuning

Tiga algoritma klasifikasi dibandingkan: **Random Forest**, **XGBoost**, dan **SVM** (dengan
`StandardScaler` di dalam pipeline, karena SVM sensitif terhadap skala fitur). Setiap model di-tuning
dengan `GridSearchCV` (5-fold cross-validation, scoring `accuracy`).


In [ ]:
PARAM_GRIDS = {
    "RandomForest": {
        "estimator": RandomForestClassifier(random_state=RANDOM_STATE),
        "param_grid": {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2],
        },
    },
    "XGBoost": {
        "estimator": XGBClassifier(
            random_state=RANDOM_STATE, use_label_encoder=False, eval_metric="mlogloss"
        ),
        "param_grid": {
            "n_estimators": [100, 200],
            "max_depth": [3, 6],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8, 1.0],
        },
    },
    "SVM": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(random_state=RANDOM_STATE)),
        ]),
        "param_grid": {
            "svm__kernel": ["rbf", "linear"],
            "svm__C": [0.1, 1, 10],
            "svm__gamma": ["scale", "auto"],
        },
    },
}

best_models = {}
best_params = {}
grid_results = {}

for name, cfg in PARAM_GRIDS.items():
    print(f"Training {name}...")
    grid = GridSearchCV(
        estimator=cfg["estimator"], param_grid=cfg["param_grid"],
        cv=5, scoring="accuracy", n_jobs=-1, verbose=0,
    )
    grid.fit(X_train, y_train_encoded)

    best_models[name] = grid.best_estimator_
    best_params[name] = grid.best_params_
    grid_results[name] = grid.cv_results_
    print(f"  Best params: {grid.best_params_}")

best_rf_model = best_models["RandomForest"]
best_xgb_model = best_models["XGBoost"]
best_svm_model = best_models["SVM"]

### 10.1 Ringkasan Hasil Grid Search

In [ ]:
def display_grid_results(model_name, grid_result, best_param):
    """Tampilkan seluruh kombinasi hyperparameter beserta skornya, kombinasi terbaik ditebalkan."""
    results_df = pd.DataFrame(grid_result)
    param_cols = [c for c in results_df.columns if c.startswith("param_")]
    display_cols = [c.replace("param_", "") for c in param_cols]

    display_df = results_df[param_cols + ["mean_test_score"]].copy()
    display_df.columns = display_cols + ["mean_test_score"]
    display_df = display_df.sort_values("mean_test_score", ascending=False).reset_index(drop=True)

    best_param_display = {k.replace("svm__", ""): v for k, v in best_param.items()}

    for col in display_cols:
        display_df[col] = display_df[col].apply(
            lambda v, c=col: f"**{v}**" if best_param_display.get(c) == v else str(v)
        )

    best_idx = display_df["mean_test_score"].astype(str).str.replace("*", "", regex=False).astype(float).idxmax()
    display_df["mean_test_score"] = display_df["mean_test_score"].apply(lambda s: f"{s:.4f}")
    display_df.loc[best_idx, "mean_test_score"] = f"**{display_df.loc[best_idx, 'mean_test_score']}**"

    print(f"\n=== {model_name} Grid Search Results ===")
    print(display_df.to_markdown(index=False))


for name in PARAM_GRIDS:
    display_grid_results(name, grid_results[name], best_params[name])

## 11. Evaluasi Model

Ketiga model dievaluasi pada data uji menggunakan metrik klasifikasi standar (accuracy, precision,
recall, macro F1-score) ditambah **Cohen's Kappa** (mengukur kesesuaian prediksi terhadap label asli
dengan koreksi terhadap kemungkinan kecocokan acak), serta confusion matrix untuk melihat pola kesalahan
antar kelas.


In [ ]:
def evaluate_classification(y_true, y_pred, model_name, class_labels):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)

    print(f"\n=== Evaluasi Model: {model_name} ===")
    print(f"Accuracy      : {acc:.4f}")
    print(f"Precision     : {prec:.4f}")
    print(f"Recall        : {rec:.4f}")
    print(f"F1-Score      : {f1:.4f}")
    print(f"Cohen's Kappa : {kappa:.4f}")
    print("-" * 40)
    print(classification_report(y_true, y_pred, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.show()

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1_macro": f1, "kappa": kappa}


class_labels = label_encoder.classes_

eval_results = {}
for name, model in best_models.items():
    preds = model.predict(X_test)
    eval_results[name] = evaluate_classification(y_test_encoded, preds, name, class_labels)

### 11.1 Perbandingan Ringkas Antar Model

In [ ]:
pd.DataFrame(eval_results).T.sort_values("f1_macro", ascending=False)

## 12. Visualisasi Peta Potensi Kawasan

Titik-titik grid diplot di atas peta dasar (basemap), diwarnai berdasarkan kelas potensi
(`potensi_kawasan`), untuk area pesisir (kedalaman > -100 m) tempat instalasi PLTPs paling realistis.


In [ ]:
gdf_potensi = gpd.GeoDataFrame(
    final_df_for_ml,
    geometry=gpd.points_from_xy(final_df_for_ml["longitude"], final_df_for_ml["latitude"]),
    crs="EPSG:4326",
)
gdf_potensi_webmercator = gdf_potensi.to_crs(epsg=3857)

# Fokus ke area pesisir (kedalaman dangkal), tempat instalasi PLTPs paling realistis
gdf_pesisir = gdf_potensi_webmercator[gdf_potensi_webmercator["depth"] > -100].copy()

fig, ax = plt.subplots(1, 1, figsize=(12, 12))
gdf_pesisir.plot(
    column="potensi_kawasan", ax=ax, legend=True, markersize=10, cmap="viridis", categorical=True,
)
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
ax.set_xlabel("Longitude (Web Mercator)")
ax.set_ylabel("Latitude (Web Mercator)")
ax.set_title("Peta Potensi Kawasan PLTPs — Sulawesi Utara")
plt.show()

## 13. Explainable AI: Permutation Feature Importance & SHAP

Dua metode interpretasi model dipakai untuk memahami fitur apa yang paling berpengaruh terhadap
klasifikasi potensi kawasan:

- **Permutation Feature Importance (PFI)** — model-agnostik, mengukur penurunan performa saat nilai
  suatu fitur diacak.
- **SHAP (SHapley Additive exPlanations)** — mengukur kontribusi tiap fitur terhadap setiap prediksi
  individual, berbasis teori permainan.


### 13.1 Permutation Feature Importance

In [ ]:
pfi_results = {}
for name, model in best_models.items():
    print(f"Menghitung PFI untuk {name}...")
    pfi_results[name] = permutation_importance(
        model, X_test, y_test_encoded, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
    )

for name, result in pfi_results.items():
    print(f"\n--- Permutation Feature Importance: {name} ---")
    print(pd.Series(result.importances_mean, index=X_test.columns).sort_values(ascending=False))

### 13.2 SHAP Values

In [ ]:
# Tree-based explainer untuk RF & XGBoost
rf_explainer = shap.Explainer(best_rf_model)
xgb_explainer = shap.Explainer(best_xgb_model)
rf_shap_values = rf_explainer(X_test)
xgb_shap_values = xgb_explainer(X_test)

# SVM bukan model pohon -> pakai KernelExplainer dengan sampel data latih sebagai background
X_train_sample = X_train.sample(n=100, random_state=RANDOM_STATE)

def svm_decision_function_wrapper(X):
    X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=X_test.columns)
    return best_svm_model.decision_function(X_df)

svm_explainer = shap.KernelExplainer(
    model=svm_decision_function_wrapper, data=X_train_sample.values,
)
print("Menghitung SHAP values untuk SVM (kernel explainer, bisa memakan waktu)...")
svm_shap_values = svm_explainer.shap_values(X_test.values)

class_names = label_encoder.classes_.tolist()
class_colors = {"Rendah": "purple", "Sedang": "teal", "Tinggi": "gold"}
cmap = mcolors.ListedColormap([class_colors[c] for c in class_names])

print("\nSHAP Summary Plot — Random Forest")
shap.summary_plot(rf_shap_values, X_test, class_names=class_names, color=cmap)

print("\nSHAP Summary Plot — XGBoost")
shap.summary_plot(xgb_shap_values, X_test, class_names=class_names, color=cmap)

print("\nSHAP Summary Plot — SVM")
shap.summary_plot(svm_shap_values, features=X_test, feature_names=X_test.columns, class_names=class_names, color=cmap)

### 13.3 Membandingkan PFI vs SHAP Antar Model

In [ ]:
def mean_abs_shap_per_feature(shap_values, n_features):
    """Rata-rata |SHAP value| per fitur, mendukung baik SHAP Explanation object maupun array NumPy."""
    if isinstance(shap_values, np.ndarray):
        return np.abs(shap_values).mean(axis=0)
    values = shap_values.values
    if values.ndim > 1 and values.shape[-1] > 1:
        return np.abs(values).mean(axis=0).mean(axis=-1)
    return np.abs(values).mean(axis=0)


pfi_rows, shap_rows = [], []
shap_values_by_model = {"RandomForest": rf_shap_values, "XGBoost": xgb_shap_values, "SVM": svm_shap_values}

for model_name, result in pfi_results.items():
    for i, feature in enumerate(X_test.columns):
        pfi_rows.append({"Feature": feature, "Importance": result.importances_mean[i],
                          "Method": "PFI", "Model": model_name})

for model_name, shap_values in shap_values_by_model.items():
    mean_abs_shap = mean_abs_shap_per_feature(shap_values, len(X_test.columns))
    for i, feature in enumerate(X_test.columns):
        shap_rows.append({"Feature": feature, "Importance": float(mean_abs_shap[i]),
                           "Method": "SHAP", "Model": model_name})

df_importance = pd.DataFrame(pfi_rows + shap_rows)
df_importance["Feature"] = df_importance["Feature"].astype("category")

g = sns.FacetGrid(df_importance, col="Model", height=6, aspect=0.8, col_wrap=3)
g.map_dataframe(
    sns.barplot, x="Importance", y="Feature", hue="Method", palette="viridis",
    hue_order=["PFI", "SHAP"], order=X_test.columns.tolist(),
)
g.add_legend()
g.fig.suptitle("Feature Importance Comparison: PFI vs. SHAP", y=1.03)
g.set_titles("{col_name}")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 14. Analisis & Diskusi Temuan

**Peringkat kepentingan fitur** yang konsisten di seluruh model dan kedua metode interpretasi
(PFI maupun SHAP):

1. **`max_tidal_speed`** — fitur paling dominan. Ini sejalan dengan teori energi kinetik arus: potensi
   energi yang dapat diekstraksi turbin arus pasang surut sebanding dengan pangkat tiga kecepatan
   arusnya, sehingga lokasi dengan arus puncak tinggi (>2–3 m/s) dianggap paling menjanjikan dalam studi
   kelayakan energi pasang surut (Lewis et al., 2017; Roc et al., 2021).
2. **`distance_to_grid_km`** — mencerminkan kelayakan ekonomi & infrastruktur. Lokasi berpotensi tinggi
   secara energi tetap kurang layak bila terlalu jauh dari gardu induk, karena biaya transmisi
   membengkak.
3. **`mean_tidal_range`** — relevan terutama untuk teknologi berbasis beda tinggi (barrage/lagoon), dan
   berkorelasi tidak langsung dengan kekuatan arus di beberapa lokasi.
4. **`depth`** — konsisten menjadi fitur paling tidak berpengaruh pada rentang data ini, meski kedalaman
   ekstrem tetap menjadi pembatas teknis dalam praktik instalasi turbin.

**PFI vs SHAP:** kedua metode sepakat pada urutan fitur, namun berbeda skala — PFI mengukur penurunan
performa saat fitur diacak (bisa sangat besar untuk fitur paling informatif), sedangkan SHAP mengukur
kontribusi rata-rata absolut terhadap prediksi individual. Perbedaan ini wajar karena keduanya
menggunakan pendekatan yang berbeda secara fundamental (perturbation-based vs. game-theoretic).

*Referensi:*
- Lewis, M. J., et al. (2017). *Tidal energy resource and technology in Australia.* Renewable and
  Sustainable Energy Reviews, 71, 358–372.
- Roc, T., Guillou, N., & Thiébaut, M. (2021). *Global assessment of tidal stream energy resource.*
  Renewable Energy, 169, 1308–1325.


## 15. Kesimpulan

- Model **XGBoost** memberikan performa klasifikasi terbaik (macro F1-score & Cohen's Kappa tertinggi)
  dibanding Random Forest dan SVM pada tugas klasifikasi 3-kelas potensi kawasan PLTPs ini.
- Analisis interpretabilitas (PFI & SHAP) menunjukkan bahwa **kecepatan arus pasang surut** adalah
  faktor paling menentukan, diikuti oleh **jarak ke infrastruktur grid listrik** — temuan yang konsisten
  dengan literatur kelayakan energi pasang surut.
- Peta potensi kawasan yang dihasilkan dapat menjadi alat bantu awal (*decision support*) untuk
  menentukan lokasi survei lanjutan instalasi PLTPs di perairan Sulawesi Utara.
